# 02 Classic models

This notebook trains and compares classical TF-IDF models for multi-label classification and regression.

In [1]:
from pathlib import Path
import sys

_PROJECT_MARKERS = (
    Path('src') / 'data' / 'prepare_dataset.py',
    Path('Turismy') / 'reviews.csv',
)


def _is_project_root(path):
    return all((path / marker).exists() for marker in _PROJECT_MARKERS)


def _candidate_roots(start):
    start = start.resolve()
    seen = set()

    for candidate in (start, *start.parents):
        if candidate not in seen:
            seen.add(candidate)
            yield candidate

    for base in (start, start.parent):
        try:
            children = list(base.iterdir())
        except OSError:
            continue

        for child in children:
            try:
                if not child.is_dir():
                    continue
                candidate = child.resolve()
            except OSError:
                continue

            if candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_project_root():
    for candidate in _candidate_roots(Path.cwd()):
        if _is_project_root(candidate):
            return candidate

    raise RuntimeError(
        'Project root was not found. Start the notebook from the repository root, '
        'the notebooks folder, or the parent folder that contains MasinskoUcenje-Projekat.'
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import json

import pandas as pd

from src.data.prepare_dataset import prepare_dataset
from src.models.train_classic import train_classification_models, save_classification_result
from src.models.train_regression import train_regression_models, save_regression_result

In [2]:
dataset_path = PROJECT_ROOT / 'Turismy' / 'reviews_enriched.csv'
if not dataset_path.exists():
    df = prepare_dataset(output_path=dataset_path)
else:
    df = pd.read_csv(dataset_path)

df.shape

(9964, 8)

In [3]:
classification_result = train_classification_models(df)
save_classification_result(classification_result)
classification_result.name, classification_result.metrics

('linear_svm',
 {'micro_precision': 0.8714842059714409,
  'micro_recall': 0.8559286017849553,
  'micro_f1': 0.8636363636363636,
  'macro_precision': 0.8123857710093,
  'macro_recall': 0.7835837069093458,
  'macro_f1': 0.7973366730919271,
  'subset_accuracy': 0.7290516808830908,
  'per_label': {'cleanliness': {'precision': 0.9436325678496869,
    'recall': 0.8845401174168297,
    'f1': 0.9131313131313131,
    'support': 511},
   'location': {'precision': 0.902127659574468,
    'recall': 0.8930075821398483,
    'f1': 0.8975444538526672,
    'support': 1187},
   'luxury': {'precision': 0.7821612349914236,
    'recall': 0.7958115183246073,
    'f1': 0.7889273356401384,
    'support': 573},
   'family_friendly': {'precision': 0.6216216216216216,
    'recall': 0.5609756097560976,
    'f1': 0.5897435897435898,
    'support': 82}}})

In [4]:
regression_result = train_regression_models(df)
save_regression_result(regression_result)
regression_result.name, regression_result.metrics

('random_forest',
 {'mae': 0.19943353596418828,
  'rmse': 0.2538271910128103,
  'r2': 0.6243062616784517})

In [5]:
metrics_dir = PROJECT_ROOT / 'artifacts' / 'metrics'
classification_metrics = json.loads((metrics_dir / 'classification_metrics.json').read_text(encoding='utf-8'))
regression_metrics = json.loads((metrics_dir / 'regression_metrics.json').read_text(encoding='utf-8'))

classification_table = pd.DataFrame(classification_metrics['models']).T
regression_table = pd.DataFrame(regression_metrics['models']).T

classification_table[['micro_f1', 'macro_f1', 'subset_accuracy']], regression_table[['mae', 'rmse', 'r2']]

(                     micro_f1  macro_f1 subset_accuracy
 logistic_regression  0.838087  0.778557        0.674862
 linear_svm           0.863636  0.797337        0.729052
 naive_bayes          0.672674  0.456013         0.43151,
                         mae      rmse        r2
 ridge              0.201658  0.250952  0.632768
 linear_regression  0.766489  1.194404 -7.318785
 random_forest      0.199434  0.253827  0.624306)